# Patrón Creacional: Builder

## Introducción
El patrón Builder permite construir objetos complejos paso a paso, permitiendo diferentes representaciones usando el mismo proceso de construcción. Es ideal cuando un objeto requiere muchos pasos de configuración o tiene muchas variantes.

## Objetivos
- Comprender el propósito y la implementación del patrón Builder.
- Identificar cuándo es útil y cuándo evitarlo.
- Comparar la solución con y sin el patrón.

## Ejemplo de la vida real
**Contexto: App de Banco**
Imagina que un banco permite a los clientes personalizar su cuenta (tipo de tarjeta, límites, servicios adicionales). El patrón Builder permite construir la cuenta paso a paso según las opciones del cliente, sin crear constructores enormes o múltiples subclases.

**¿Dónde se usa en proyectos reales?**
En la construcción de objetos complejos como configuraciones de cuentas bancarias, generación de reportes, construcción de interfaces gráficas, etc.

## Sin patrón Builder (forma errónea)
El código cliente debe conocer todos los detalles de construcción. Esto puede llevar a constructores con muchos parámetros y código difícil de mantener.

In [1]:
class Sandwich:
    def __init__(self, pan: str, carne: str, vegetales: str) -> None:
        self.pan = pan
        self.carne = carne
        self.vegetales = vegetales
    def mostrar(self) -> None:
        print(f'Sandwich de {self.pan}, {self.carne}, {self.vegetales}')

sandwich = Sandwich('blanco', 'pollo', 'lechuga')
sandwich.mostrar()

Sandwich de blanco, pollo, lechuga


## Con patrón Builder (forma correcta)
El cliente utiliza un objeto builder para construir el producto paso a paso, sin preocuparse por los detalles internos. Esto facilita la personalización y el mantenimiento.

In [2]:
class SandwichBuilder:
    def __init__(self) -> None:
        self.pan: str | None = None
        self.carne: str | None = None
        self.vegetales: str | None = None

    def set_pan(self, pan: str) -> 'SandwichBuilder':
        self.pan = pan
        return self
    def set_carne(self, carne: str) -> 'SandwichBuilder':
        self.carne = carne
        return self
    def set_vegetales(self, vegetales: str) -> 'SandwichBuilder':
        self.vegetales = vegetales
        return self
    def build(self) -> Sandwich:
        return Sandwich(self.pan, self.carne, self.vegetales)

class SandwichMother:
    def __init__(self, builder: SandwichBuilder) -> None:
        self.builder = builder
    def construir_club(self, carne: str) -> Sandwich:
        return (self.builder.set_pan('blanco')
                .set_carne(carne)
                .build())
    def construir_italiano(self) -> Sandwich:
        return (self.builder.set_pan('integral')
                .set_carne('jamón')
                .set_vegetales('tomate')
                .build())

builder = SandwichBuilder()
sandwich = builder.set_pan('blanco').set_carne('pollo').set_vegetales('lechuga').build()
sandwich.mostrar()

mother = SandwichMother(SandwichBuilder())
sandwich = mother.construir_club('pollo')
sandwich.mostrar()

Sandwich de blanco, pollo, lechuga
Sandwich de blanco, pollo, None


## UML del patrón Builder
```plantuml
@startuml
class Sandwich {
    + __init__(pan, carne, vegetales)
    + mostrar()
}
class SandwichBuilder {
    + set_pan(pan)
    + set_carne(carne)
    + set_vegetales(vegetales)
    + build()
}
SandwichBuilder ..> Sandwich
@enduml
```

## Otro ejemplo de la vida real: Constructor de consultas SQL (Query Builder)
**Contexto:** las capas de acceso a datos rara vez arman un `SELECT` con todas sus cláusulas de una vez. Normalmente se van agregando condiciones de forma condicional a lo largo del código (un filtro si el usuario buscó algo, otro si aplicó un filtro de fecha, orden solo si lo pidió, límite solo en paginación). Esto es literalmente lo que hacen por debajo ORMs como el de Django o SQLAlchemy con sus *query builders* encadenables.

### Sin patrón (forma errónea)
Una función con muchos parámetros opcionales y concatenación manual de strings: crece sin control cada vez que se agrega una cláusula nueva (JOIN, GROUP BY, HAVING...).

In [3]:
def construir_sql(
    tabla: str,
    columnas: list[str],
    condicion: str | None = None,
    orden: str | None = None,
    limite: int | None = None,
) -> str:
    sql = f"SELECT {', '.join(columnas)} FROM {tabla}"
    if condicion:
        sql += f" WHERE {condicion}"
    if orden:
        sql += f" ORDER BY {orden}"
    if limite:
        sql += f" LIMIT {limite}"
    return sql

# Cada nueva cláusula (JOIN, GROUP BY...) obliga a agregar otro parámetro y otro `if`
print(construir_sql('usuarios', ['id', 'nombre'], condicion='activo = true', orden='nombre', limite=10))

SELECT id, nombre FROM usuarios WHERE activo = true ORDER BY nombre LIMIT 10


### Con patrón (forma correcta)
Cada cláusula se agrega solo si el código la necesita, en el orden que sea, con métodos encadenables (`.donde()`, `.ordenar_por()`...). El objeto final (`build()`) solo incluye lo que realmente se configuró.

In [4]:
class QueryBuilder:
    def __init__(self, tabla: str) -> None:
        self._tabla = tabla
        self._columnas: list[str] = ['*']
        self._condiciones: list[str] = []
        self._orden: str | None = None
        self._limite: int | None = None

    def seleccionar(self, *columnas: str) -> 'QueryBuilder':
        self._columnas = list(columnas)
        return self

    def donde(self, condicion: str) -> 'QueryBuilder':
        self._condiciones.append(condicion)
        return self

    def ordenar_por(self, columna: str) -> 'QueryBuilder':
        self._orden = columna
        return self

    def limitar(self, n: int) -> 'QueryBuilder':
        self._limite = n
        return self

    def build(self) -> str:
        sql = f"SELECT {', '.join(self._columnas)} FROM {self._tabla}"
        if self._condiciones:
            sql += f" WHERE {' AND '.join(self._condiciones)}"
        if self._orden:
            sql += f" ORDER BY {self._orden}"
        if self._limite:
            sql += f" LIMIT {self._limite}"
        return sql


query = (QueryBuilder('usuarios')
         .seleccionar('id', 'nombre')
         .donde('activo = true')
         .donde("pais = 'CO'")
         .ordenar_por('nombre')
         .limitar(10)
         .build())
print(query)

SELECT id, nombre FROM usuarios WHERE activo = true AND pais = 'CO' ORDER BY nombre LIMIT 10


### UML del ejemplo de Query Builder
```plantuml
@startuml
class QueryBuilder {
    - _tabla
    - _columnas
    - _condiciones
    - _orden
    - _limite
    + seleccionar(columnas)
    + donde(condicion)
    + ordenar_por(columna)
    + limitar(n)
    + build()
}
@enduml
```

### ¿Dónde más se usa Builder?
- **ORMs y query builders:** `QuerySet` de Django, `session.query()` de SQLAlchemy o el builder de Knex.js encadenan `.filter()`, `.order_by()`, `.limit()` exactamente así.
- **Clientes HTTP configurables:** construir una petición (`RequestBuilder`) agregando headers, query params, body y timeout paso a paso antes de `.send()`.
- **Generación de documentos:** un `PDFBuilder` o `ReporteBuilder` que agrega secciones (portada, tabla, gráfico, pie de página) solo si el reporte las necesita.
- **Configuración de contenedores/infraestructura:** un `DockerfileBuilder` o el builder fluido de un `Pipeline` de CI/CD que agrega pasos condicionalmente.
- **Formularios dinámicos en frontend:** un `FormBuilder` que agrega campos (texto, select, fecha) según la configuración del formulario, sin un constructor gigante.

**Ejercicio de reflexión:** agrega un método `.agrupar_por(columna)` (GROUP BY) a `QueryBuilder`. ¿Cuántas líneas tuviste que tocar? Compáralo con lo que habría costado agregarlo a `construir_sql()`.

## Actividad
Crea tu propio Builder para construir un objeto complejo, como una pizza o un computador.

---

## Explicación de conceptos clave
- **Separación de construcción y representación:** El Builder separa la lógica de construcción de la representación final del objeto.
- **Flexibilidad:** Permite crear diferentes representaciones del mismo objeto usando el mismo proceso de construcción.
- **Aplicación en la vida real:** Útil en sistemas donde los objetos tienen muchas opciones de configuración, como cuentas bancarias, reportes o interfaces.

## Conclusión
El patrón Builder es ideal para construir objetos complejos de manera controlada y flexible. Facilita la personalización y el mantenimiento, y es ampliamente usado en sistemas bancarios, generación de documentos y aplicaciones con configuraciones avanzadas.